In [1]:
import cv2
import mediapipe as mp
import pandas as pd
import os

# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, 
                    min_detection_confidence=0.5, 
                    min_tracking_confidence=0.5)

def extract_kinematics(video_path, output_csv):
    print(f"Processing: {video_path}")
    
    if not os.path.exists(video_path):
        print(f"Error: Could not find {video_path}. Check your file name and folder.")
        return
        
    cap = cv2.VideoCapture(video_path)
    data = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break 

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)

        # Extract coordinates
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            
            row = {
                'frame': frame_idx,
                'hip_x': landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x,
                'hip_y': landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y,
                'hip_z': landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].z,
                'knee_x': landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x,
                'knee_y': landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y,
                'knee_z': landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].z,
                'ankle_x': landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x,
                'ankle_y': landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y,
                'ankle_z': landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].z,
            }
            data.append(row)
        frame_idx += 1

    cap.release()
    
    df = pd.DataFrame(data)
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Success! Saved {len(df)} frames of data to {output_csv}\n")

# Run the extraction
extract_kinematics('../data/raw/normal_gait_01.mp4', '../data/processed/normal_gait_01.csv')
extract_kinematics('../data/raw/stiff_knee_01.mp4', '../data/processed/stiff_knee_01.csv')

Processing: ../data/raw/normal_gait_01.mp4
Success! Saved 185 frames of data to ../data/processed/normal_gait_01.csv

Processing: ../data/raw/stiff_knee_01.mp4
Success! Saved 321 frames of data to ../data/processed/stiff_knee_01.csv

